# COSMoS BC Parent Resolution

Self-join `Parent_Label` → `BC_Name` across all BCs in `COSMoS_BC_DSS.xlsx`.

**Question.** Does every `Parent_Label` resolve to a known BC? How deep is the hierarchy per domain?

**Framing (post Linda's input).** Direct parent (`Parent_Label` / `Hierarchy_Path`) and `Categories` carry different relations by design. The direct parent encodes structural ancestry. `Categories` is the COSMoS working group's confirmed mechanism for instrument-level grouping and other cross-cutting tags. This notebook does not treat `Categories`/parent mismatch as a defect; it profiles the two as independent views.


In [ ]:
import pandas as pd
from pathlib import Path

# Directory structure - notebook is in notebooks/ subfolder
BASE_DIR    = Path.cwd().parent
INTERIM_DIR = BASE_DIR / 'interim'
INTERIM_FILE = INTERIM_DIR / 'COSMoS_BC_DSS.xlsx'

assert INTERIM_FILE.exists(), f'Interim file not found: {INTERIM_FILE}'

df = pd.read_excel(INTERIM_FILE, sheet_name='BC_DSS')
print(f'BC_DSS rows: {len(df)}')
print(f'Unique BCs:  {df["BC_Name"].nunique()}')

## BC-level frame

Dedupe on `BC_Name`. Each BC contributes one row for structural analysis regardless of how many DSSs it has.


In [ ]:
bc_cols = ['BC_Name', 'COSMoS_BC_ID', 'NCIt_Code', 'BC_Type',
           'Categories', 'Hierarchy_Path', 'Parent_Label', 'Domains_with_DS']
bc = df.drop_duplicates('BC_Name')[bc_cols].copy()
print(f'Distinct BCs: {len(bc)}')

## Parent_Label → BC_Name resolution


In [ ]:
bc_names = set(bc['BC_Name'].dropna())
has_parent = bc['Parent_Label'].notna()
bc.loc[has_parent, 'parent_resolves'] = bc.loc[has_parent, 'Parent_Label'].isin(bc_names)

total_with_parent = int(has_parent.sum())
resolved = int(bc['parent_resolves'].fillna(False).sum())
print(f'BCs with Parent_Label set : {total_with_parent}')
print(f'BCs with parent resolved  : {resolved}')
print(f'Unresolved                : {total_with_parent - resolved}')

## Hierarchy depth

Depth computed from `Hierarchy_Path` as the number of `>`-separated segments. Depth 0 = BCs with no `Hierarchy_Path` (roots or isolated).


In [ ]:
def depth(p):
    if pd.isna(p):
        return 0
    return len([x for x in str(p).split('>') if x.strip()])

bc['depth'] = bc['Hierarchy_Path'].apply(depth)
print(bc['depth'].value_counts().sort_index().to_string())

## Depth per domain

A BC can carry DSSs in multiple domains. The `Domains_with_DS` column captures this as a semicolon-separated list. Profile depth per domain by exploding the list.


In [ ]:
def split_domains(s):
    if pd.isna(s):
        return []
    return [x.strip() for x in str(s).split(';') if x.strip()]

bc_dom = bc.assign(Domain=bc['Domains_with_DS'].apply(split_domains)).explode('Domain')
bc_dom = bc_dom[bc_dom['Domain'].notna() & (bc_dom['Domain'] != '')]

depth_by_domain = (bc_dom.groupby('Domain')['depth']
                         .agg(['count', 'min', 'max', 'mean'])
                         .round(2)
                         .sort_values('count', ascending=False))
print(depth_by_domain.to_string())

## Categories vs parent chain

Walk the `Parent_Label` chain for each BC. For every BC, compare its `Categories` tokens against the set of ancestors in the direct chain. Count Category tokens that are themselves `BC_Name` values but not on the direct chain.

**Interpretation.** Off-chain Category → BC edges are not errors. They are the working group's cross-cutting grouping mechanism: the same BC tagged from outside its structural ancestry , an instrument family reference, a clinical use-case tag, a discovery alias.


In [ ]:
parent_map = dict(zip(bc['BC_Name'], bc['Parent_Label']))

def chain(name):
    out, seen = [], set()
    cur = parent_map.get(name)
    while pd.notna(cur) and cur not in seen:
        seen.add(cur)
        out.append(cur)
        cur = parent_map.get(cur)
    return out

bc['ParentChain'] = bc['BC_Name'].apply(chain)

def split_cats(c):
    if pd.isna(c):
        return []
    return [x.strip() for x in str(c).split(';') if x.strip()]

def cats_offchain(row):
    cats = split_cats(row['Categories'])
    chain_set = set(row['ParentChain'])
    return [c for c in cats if c in bc_names and c not in chain_set and c != row['BC_Name']]

bc['Categories_BCs_offchain'] = bc.apply(cats_offchain, axis=1)
bc['offchain_count'] = bc['Categories_BCs_offchain'].apply(len)

print(f'BCs with at least one off-chain Category→BC edge : {(bc["offchain_count"] > 0).sum()}')
print(f'Total off-chain Category→BC edges                : {bc["offchain_count"].sum()}')

## Sample off-chain edges

Show a handful so the reader sees what the pattern looks like in practice.


In [ ]:
sample = bc[bc['offchain_count'] > 0].head(10)
for _, r in sample.iterrows():
    print(f'BC            : {r["BC_Name"]}')
    print(f'Hierarchy     : {r["Hierarchy_Path"]}')
    print(f'ParentChain   : {r["ParentChain"]}')
    print(f'Off-chain cats: {r["Categories_BCs_offchain"]}')
    print()

## Summary

- Every `Parent_Label` resolves to a `BC_Name` in the same file (100% self-consistent).
- Hierarchy depth ranges 0 to 4. The bulk of BCs sit at depth 1.
- `Categories` carries a substantial number of edges to BCs outside the direct parent chain. By Linda's confirmation this is intentional: `Categories` is the grouping mechanism, distinct from structural ancestry.

Both columns should be preserved and queried independently, not reconciled.
